# L6 Lab: Outliers & Noise Detection

**Goal:** Detect outliers with IQR and Z-score, decide keep / cap / remove, and document the impact.

**Dataset:** `../datasets/sample_sales.csv` or `sample_accidents.csv`


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../datasets/sample_sales.csv")
print(df.shape)
print(df.head())
print(df.describe())


## 1. IQR method
Outliers often sit below Q1 − 1.5×IQR or above Q3 + 1.5×IQR.


In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

num_col = [c for c in df.select_dtypes("number").columns if c.lower() not in ("id", "year", "month")][0]
lo, hi = iqr_bounds(df[num_col])
print(f"Column: {num_col}  bounds: [{lo:.2f}, {hi:.2f}]")
mask_iqr = (df[num_col] < lo) | (df[num_col] > hi)
print("IQR outliers:", mask_iqr.sum())


## 2. Z-score method
Flag |z| > 3 as extreme (rule of thumb).


In [ ]:
z = (df[num_col] - df[num_col].mean()) / df[num_col].std(ddof=0)
mask_z = z.abs() > 3
print("Z-score outliers (|z|>3):", mask_z.sum())
df.assign(z=z).loc[mask_z, [num_col]].head()


## 3. Treatment
- **Keep** true rare events
- **Cap** at bounds for stability
- **Remove** only if clear data error

### Exercises
1. Apply IQR and Z-score to two numeric columns; compare counts.
2. Cap one column at IQR bounds; compare mean/std before vs after.
3. Write 3 sentences: which method you prefer for this dataset and why.


In [ ]:
df_capped = df.copy()
df_capped[num_col] = df_capped[num_col].clip(lo, hi)
print("Before mean/std:", df[num_col].mean(), df[num_col].std())
print("After  mean/std:", df_capped[num_col].mean(), df_capped[num_col].std())
